In [ ]:
!pip install -U langchain langchain-core==1.0 langchain-community langchain-experimental langchain_google_genai

  Using cached langchain-1.4.9-py3-none-any.whl.metadata (6.2 kB)
  Using cached langchain_community-0.4.2-py3-none-any.whl.metadata (3.4 kB)
  Using cached langchain_experimental-0.4.2-py3-none-any.whl.metadata (1.6 kB)
  Using cached langchain_google_genai-4.4.3-py3-none-any.whl.metadata (2.7 kB)
INFO: pip is looking at multiple versions of langchain to determine which version is compatible with other requirements. This could take a while.
  Using cached langchain-1.4.8-py3-none-any.whl.metadata (6.2 kB)
  Using cached langchain-1.4.7-py3-none-any.whl.metadata (6.2 kB)
  Using cached langchain-1.4.6-py3-none-any.whl.metadata (6.2 kB)
  Using cached langchain-1.4.5-py3-none-any.whl.metadata (6.2 kB)
  Using cached langchain-1.4.4-py3-none-any.whl.metadata (6.2 kB)
  Using cached langchain-1.4.3-py3-none-any.whl.metadata (6.2 kB)
  Using cached langchain-1.4.2-py3-none-any.whl.metadata (6.2 kB)
INFO: pip is still looking at multiple versions of langchain to determine which version is c

### Langchain Programming including setup in Cell 1

In [ ]:
import os
from langchain_google_genai import ChatGoogleGenerativeAI
from google.colab import userdata

os.environ["GEMINI_API_KEY"]=userdata.get('gemini_key')

chat = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite")


Prompt Templates

In [ ]:
#Prompts
from langchain_core.prompts import PromptTemplate

# Simple prompt with placeholders
prompt_template = PromptTemplate.from_template(
    "Tell me a {adjective} joke about {content}."
)

# Filling placeholders to create a prompt
filled_prompt = prompt_template.format(adjective="funny", content="robots")
print(filled_prompt)
response = chat.invoke(filled_prompt)
print(response.content)

Tell me a funny joke about robots.
[{'type': 'text', 'text': 'Why do robots never panic?\n\nBecause they have **nerves of steel**! \n\n*(...and if things really go wrong, they just reboot.)*', 'extras': {'signature': 'EmAKXgFpFH0TmNe0WKFRWNDMIni9b62sMG134t/dawv98HS/zXPWNdyn8rjyZ4r2J5U2sT6dZ/lUjEneKc18n/ZJ8vO7kzjs+J9qJVs1pKNc6uBMmKHeQKfoltZm05bmC90='}}]


### Using Memory and User sessions - Useful especially for Chatbots

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
model=ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite")
model

ChatGoogleGenerativeAI(output_version=None, model='models/gemini-3.5-flash-lite', google_api_key=SecretStr('**********'), client=<google.ai.generativelanguage_v1beta.services.generative_service.client.GenerativeServiceClient object at 0x78851bfedf90>, default_metadata=(), model_kwargs={})

In [ ]:
model.invoke("Hi , My name is Alex and I am an AI Engineer")

AIMessage(content=[{'type': 'text', 'text': "Hi Alex! It's great to meet you. \n\nAs an AI engineer, you must be working on some fascinating stuff right now given how fast the field is moving. What kind of projects are you focusing on lately? Are you working on LLMs, computer vision, MLOps, or something else entirely? \n\nHow can I help you today?", 'extras': {'signature': 'EmAKXgFpFH0TtDiIyf2ESVZBOlh51XWG0f/uTxuuQc2/wMZ+Aywf9Bk+iEDlADaHxIdrs1GsFsV2h7hB4VjVN+0tsmdyLl+h2pbArBLAEhF49Z+F/v6285RgiUs9c7v++Z0='}}], additional_kwargs={}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--2b6dabca-2a4c-4d27-8127-ca6a89c4c68c-0', usage_metadata={'input_tokens': 13, 'output_tokens': 74, 'total_tokens': 87, 'input_token_details': {'cache_read': 0}})

In [ ]:
model.invoke("What is my name")

AIMessage(content=[{'type': 'text', 'text': "I don't know your name! You haven't told me yet. What is it?", 'extras': {'signature': 'EmAKXgFpFH0TmTa43tcpBUXLOBHKeknrdR4gd0+eO/R4Dim0IaE7Cs9+np31KKdPzBJ4CHoavwY7F9ptFXMUw3L7cWJi1u+EFDVadvaR1FG7zFXiYkYYj31bCCBkKBfggDM='}}], additional_kwargs={}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--7e958c52-20b9-4b50-a692-6c3ceeff59f0-0', usage_metadata={'input_tokens': 5, 'output_tokens': 20, 'total_tokens': 25, 'input_token_details': {'cache_read': 0}})

In [ ]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

store={}

def get_session_history(session_id:str)->BaseChatMessageHistory:
    if session_id not in store:
        store[session_id]=ChatMessageHistory()
    return store[session_id]

with_message_history=RunnableWithMessageHistory(model,get_session_history)

In [ ]:
#Long term memory and state
!pip install sqlalchemy[asyncio]

from langchain_community.chat_message_histories import SQLChatMessageHistory

def get_sqlite_history(session_id: str):
    return SQLChatMessageHistory(
        session_id=session_id,
        connection_string="sqlite:///sqlite_chat_history.db"
    )

# Wrap the model with SQLite-backed message history
persistent_chat_history = RunnableWithMessageHistory(
    model,
    get_sqlite_history
)

# Let's test it with a session
config_sql = {"configurable": {"session_id": "sqlite_session_1"}}

# First interaction
response1 = persistent_chat_history.invoke(
    [HumanMessage(content="Hi, my name is Sarah and I am learning AI.")],
    config=config_sql
)
print("Response 1:", response1.content)

# Second interaction (recalling the name from SQLite)
response2 = persistent_chat_history.invoke(
    [HumanMessage(content="What is my name?")],
    config=config_sql
)
print("Response 2:", response2.content)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 623.8/623.8 kB 7.1 MB/s  0:00:00


/usr/local/lib/python3.13/dist-packages/langchain_core/runnables/history.py:585: LangChainDeprecationWarning: `connection_string` was deprecated in LangChain 0.2.2 and will be removed in 1.0. Use connection instead.
  message_history = self.get_session_history(


Response 1: [{'type': 'text', 'text': "Hi Sarah! It’s great to meet you, and welcome to the world of AI! \n\nIt's a fascinating, fast-moving field, and there's never been a better time to dive in. Whether you're interested in how ChatGPT works, coding machine learning models, or just want to understand the basics, I'm here to help.\n\nTo help me tailor things for you: **What got you interested in AI, and where are you starting from (complete beginner, some coding experience, etc.)?**", 'extras': {'signature': 'EmAKXgFpFH0T91OeuLRt0aYyBCDQ1bAYcqC5tqy5ekYWQ21cxXaUMN1YXXgU1NWlRE9Nj2XmKpn+i2hBl80U8d6l9SZHqcrgBhlf1+OfcRGBbisiC+osjgZW1QKG9+3Z50A='}}]
Response 2: [{'type': 'text', 'text': 'Your name is Sarah!', 'extras': {'signature': 'EmAKXgFpFH0TiFxkJ9vOWVRh2tAjSsw6ZZmJ/O36kmd1yRyJXZD7anZA2ESWK1Z2Dogj+7bUyqFL5i0ESxXetonOmP6WkGyEomnCBO84A8X6/P1ZHQYPYrapL2YcvbjmuG4='}}]


In [ ]:
config={"configurable":{"session_id":"chat1"}}

In [ ]:
from langchain_core.messages import HumanMessage
from langchain_core.messages import AIMessage

response=with_message_history.invoke(
    [HumanMessage(content="Hi , My name is Alex and I am an AI Engineer")],
    config=config
)

In [ ]:
response.content

[{'type': 'text',
  'text': "Hi Alex! It's great to meet you. \n\nAs an AI engineer, you must be working on some fascinating stuff right now given how fast the field is moving. What kind of projects do you usually focus on—are you working on LLMs, computer vision, MLOps, or something else entirely? \n\nHow can I help you out today?",
  'extras': {'signature': 'EmAKXgFpFH0T7IrRRsr4elN6Y4bgA8Ybxolk8ZPROAdjaq7zW4eDU8XCaO5sQxXd+XoiFYMzxt1kvOpFQRllJJBdRNf72N8eE+R7qx7hB6RZEi4YIsITZZ+XwxTlv8JHe6k='}}]

In [ ]:
with_message_history.invoke(
    [HumanMessage(content="What's my name?")],
    config=config,
)

AIMessage(content=[{'type': 'text', 'text': 'Your name is Alex!', 'extras': {'signature': 'EmAKXgFpFH0TTlANYvMQWaxsDq9oL68QfCZ+O2v2YkU6XLQWHLoMM6gIfQlOjw0lhjohYpwRpy1nKcjYj0dlTAdoNyu6QQIYRQFKDkAUz6rpQG9GnLeseYWhz9fFz+VPGBs='}}], additional_kwargs={}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--bc12e7d2-cd89-4497-af84-d2ba49d92794-0', usage_metadata={'input_tokens': 96, 'output_tokens': 5, 'total_tokens': 101, 'input_token_details': {'cache_read': 0}})

In [ ]:
## change the config-->session id
config1={"configurable":{"session_id":"chat2"}}
response=with_message_history.invoke(
    [HumanMessage(content="Whats my name")],
    config=config1
)
response.content

In [ ]:
response=with_message_history.invoke(
    [HumanMessage(content="Hey My name is John")],
    config=config1
)
response.content

In [ ]:
response=with_message_history.invoke(
    [HumanMessage(content="Whats my name")],
    config=config1
)
response.content

**Using Prompt templates and LCEL with History**


*Understand and Practice on your own*

---



In [ ]:
from langchain_core.prompts import ChatPromptTemplate,MessagesPlaceholder
prompt=ChatPromptTemplate.from_messages(
    [
        ("system","You are a helpful assistant.Answer all the question to the best of your ability"),
        MessagesPlaceholder(variable_name="messages")
    ]
)

chain=prompt|model

In [ ]:
chain.invoke({"messages":[HumanMessage(content="Hi My name is Amit")]})

AIMessage(content=[{'type': 'text', 'text': "Hello Amit! It's nice to meet you. How can I help you today?", 'extras': {'signature': 'EmAKXgFpFH0TWoUCTrWDj4geyqCFuN6F+yPfCjTrLbm2MkaR1f9hSDMNDt3qJviSaBi+myTCAX44SSHNPgIuv3gJ9V2fl2tsVROt568dLzZUTnEc1YXUC0csy8aaIp8cU0c='}}], additional_kwargs={}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--f65bcdaa-f5db-4ea0-bcf0-58f089f9b347-0', usage_metadata={'input_tokens': 22, 'output_tokens': 18, 'total_tokens': 40, 'input_token_details': {'cache_read': 0}})

In [ ]:
chain.invoke({"messages":[HumanMessage(content="What's my name?")]})

AIMessage(content=[{'type': 'text', 'text': "I don't know your name yet! What should I call you?", 'extras': {'signature': 'EmAKXgFpFH0T4GKyDeQ0hEk4jSyBqi7KSOyf3/ybPZW4sHh82jDPoOHyavhaSmfSBxOMXlEpp1eHqewh1ARsiJzr269lb7JgqGk0Iymv7gEhZ9/mciFwPA4trnmNNmT7i4k='}}], additional_kwargs={}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--8695a9cb-ac22-4d7a-9cbd-5583a146b6f2-0', usage_metadata={'input_tokens': 23, 'output_tokens': 15, 'total_tokens': 38, 'input_token_details': {'cache_read': 0}})

In [ ]:
with_message_history=RunnableWithMessageHistory(chain,get_session_history)

In [ ]:
config = {"configurable": {"session_id": "chat3"}}
response=with_message_history.invoke(
    [HumanMessage(content="Hi My name is Jack")],
    config=config
)

response

In [ ]:
response = with_message_history.invoke(
    [HumanMessage(content="What's my name?")],
    config=config,
)

response.content

In [ ]:
## Add more complexity

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a helpful assistant. Answer all questions to the best of your ability in {language}.",
        ),
        MessagesPlaceholder(variable_name="messages"),
    ]
)

chain = prompt | model

In [ ]:
response=chain.invoke({"messages":[HumanMessage(content="Hi My name is Jack")],"language":"French"})
response.content

Let's now wrap this more complicated chain in a Message History class. This time, because there are multiple keys in the input, we need to specify the correct key to use to save the chat history.

In [ ]:
with_message_history=RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="messages"
)

In [ ]:
config = {"configurable": {"session_id": "chat4"}}
repsonse=with_message_history.invoke(
    {'messages': [HumanMessage(content="Hi,I am Jack")],"language":"French"},
    config=config
)
repsonse.content

In [ ]:
response = with_message_history.invoke(
    {"messages": [HumanMessage(content="whats my name?")], "language": "Hindi"},
    config=config,
)

response.content